In [ ]:
import sys
sys.path.append('..')

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from simulation.create_network import create_microgrid
from simulation.run_powerflow import run_powerflow_analysis, check_constraints
from simulation.scenario_generator import generate_scenarios, create_time_series_scenarios
from simulation.visualization import plot_results, plot_constraints, plot_scenario_comparison

np.random.seed(42)

## 1. Create Base Network and Run Power Flow

In [ ]:
# Create a mesh microgrid
base_graph, meta = create_microgrid(num_nodes=10, topology='mesh', seed=42)
print(f"Base network: {meta['num_nodes']} nodes, {meta['num_edges']} edges")

In [ ]:
# Run power flow analysis
results = run_powerflow_analysis(base_graph, max_iterations=100, tolerance=1e-6)
print(f"\nPower Flow Results:")
print(f"  Converged: {results['converged']}")
print(f"  Iterations: {results['iterations']}")
print(f"  Total Loss: {results['total_loss']:.6f} MW")
print(f"  Voltage Range: {results['voltages'].min():.4f} - {results['voltages'].max():.4f} pu")

## 2. Visualize Power Flow Results

In [ ]:
# Plot detailed power flow results
fig, axes = plot_results(results, figsize=(15, 10))
plt.tight_layout()
plt.show()

## 3. Check Constraints

In [ ]:
# Check for constraint violations
violations = check_constraints(base_graph, results)

print(f"Constraint Violations:")
print(f"  Voltage violations: {len(violations['voltage_violations'])}")
print(f"  Thermal violations: {len(violations['thermal_violations'])}")
print(f"  Frequency violations: {len(violations['frequency_violations'])}")

if violations['voltage_violations']:
    print(f"\nVoltage Violations:")
    for v in violations['voltage_violations'][:3]:
        print(f"  Node {v['node']}: {v['voltage']:.4f} pu (limits: {v['limits']})")

In [ ]:
# Visualize constraint violations
fig, axes = plot_constraints(violations, figsize=(14, 5))
plt.tight_layout()
plt.show()

## 4. Generate Scenarios

In [ ]:
# Generate renewable variation scenarios
renewable_scenarios = generate_scenarios(
    base_graph,
    num_scenarios=5,
    scenario_type='renewable_variation',
    seed=42
)
print(f"Generated {len(renewable_scenarios)} renewable variation scenarios")

# Run power flow for each scenario
renewable_results = []
for i, scenario in enumerate(renewable_scenarios):
    result = run_powerflow_analysis(scenario)
    renewable_results.append(result)
    print(f"  Scenario {i+1}: Total Loss = {result['total_loss']:.6f} MW")

In [ ]:
# Generate load variation scenarios
load_scenarios = generate_scenarios(
    base_graph,
    num_scenarios=5,
    scenario_type='load_variation',
    seed=42
)
print(f"Generated {len(load_scenarios)} load variation scenarios")

# Run power flow for each scenario
load_results = []
for i, scenario in enumerate(load_scenarios):
    result = run_powerflow_analysis(scenario)
    load_results.append(result)
    print(f"  Scenario {i+1}: Total Loss = {result['total_loss']:.6f} MW")

In [ ]:
# Generate contingency scenarios
contingency_scenarios = generate_scenarios(
    base_graph,
    num_scenarios=5,
    scenario_type='contingency',
    seed=42
)
print(f"Generated {len(contingency_scenarios)} contingency scenarios")

# Run power flow for each scenario
contingency_results = []
for i, scenario in enumerate(contingency_scenarios):
    result = run_powerflow_analysis(scenario)
    contingency_results.append(result)
    print(f"  Scenario {i+1}: Total Loss = {result['total_loss']:.6f} MW, Converged = {result['converged']}")

## 5. Compare Scenarios

In [ ]:
# Compare losses across scenario types
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

scenarios_data = [
    ('Renewable Variation', renewable_results),
    ('Load Variation', load_results),
    ('Contingency', contingency_results)
]

for idx, (title, results_list) in enumerate(scenarios_data):
    losses = [r['total_loss'] for r in results_list]
    axes[idx].bar(range(len(losses)), losses, color='steelblue', alpha=0.7)
    axes[idx].set_xlabel('Scenario')
    axes[idx].set_ylabel('Total Loss (MW)')
    axes[idx].set_title(title)
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Time Series Analysis

In [ ]:
# Create time series scenarios (24 hours)
time_series = create_time_series_scenarios(
    base_graph,
    num_timesteps=24
)
print(f"Generated {len(time_series)} time steps")

# Run power flow for each time step
ts_results = []
for t, scenario in time_series:
    result = run_powerflow_analysis(scenario)
    ts_results.append(result)

In [ ]:
# Plot time series analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total losses over time
losses = [r['total_loss'] for r in ts_results]
axes[0, 0].plot(losses, 'o-', linewidth=2, markersize=6)
axes[0, 0].set_xlabel('Hour')
axes[0, 0].set_ylabel('Total Loss (MW)')
axes[0, 0].set_title('Total Losses Over 24 Hours')
axes[0, 0].grid(True, alpha=0.3)

# Average voltage over time
avg_voltages = [np.mean(r['voltages']) for r in ts_results]
axes[0, 1].plot(avg_voltages, 'o-', linewidth=2, markersize=6, color='orange')
axes[0, 1].axhline(y=1.0, color='r', linestyle='--', alpha=0.5)
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Average Voltage (pu)')
axes[0, 1].set_title('Average Voltage Over 24 Hours')
axes[0, 1].grid(True, alpha=0.3)

# Max voltage over time
max_voltages = [np.max(r['voltages']) for r in ts_results]
min_voltages = [np.min(r['voltages']) for r in ts_results]
axes[1, 0].fill_between(range(len(max_voltages)), min_voltages, max_voltages, alpha=0.3)
axes[1, 0].plot(max_voltages, 'o-', linewidth=2, label='Max')
axes[1, 0].plot(min_voltages, 'o-', linewidth=2, label='Min')
axes[1, 0].axhline(y=1.05, color='r', linestyle='--', alpha=0.5, label='Upper Limit')
axes[1, 0].axhline(y=0.95, color='r', linestyle='--', alpha=0.5, label='Lower Limit')
axes[1, 0].set_xlabel('Hour')
axes[1, 0].set_ylabel('Voltage (pu)')
axes[1, 0].set_title('Voltage Range Over 24 Hours')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Statistics
axes[1, 1].text(0.5, 0.8, 'Time Series Statistics', ha='center', fontsize=12, fontweight='bold')
axes[1, 1].text(0.1, 0.6, f"Avg Loss: {np.mean(losses):.4f} MW", fontsize=10)
axes[1, 1].text(0.1, 0.5, f"Max Loss: {np.max(losses):.4f} MW", fontsize=10)
axes[1, 1].text(0.1, 0.4, f"Min Loss: {np.min(losses):.4f} MW", fontsize=10)
axes[1, 1].text(0.1, 0.25, f"Avg Voltage: {np.mean(avg_voltages):.4f} pu", fontsize=10)
axes[1, 1].text(0.1, 0.15, f"Voltage Std Dev: {np.std(avg_voltages):.6f} pu", fontsize=10)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()